# 0825_peace_012_recall_aligned_model_comparison

동일 Recall 목표(0.95/0.97/0.99)에서 세 모델 전략의 FP/FCR을 비교하고, walk-forward 미래 구간과 final retrospective Test를 함께 정리합니다.

In [1]:

import gc
import hashlib
import json
import logging
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = '0825_peace_012_recall_aligned_model_comparison'
RANDOM_STATE = 42
TARGET = 'class'
TIME_COLUMN = 'timestamp'
TYPE_COLUMN = 'inspection_type'
RECORD_ID = 'record_id'
DECISION_THRESHOLD = 0.5
TARGET_RECALLS = [0.95, 0.97, 0.99]
DIAGNOSTIC_TARGET = 0.97
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {
    'fold_1': [0.30],
    'fold_2': [0.30, 0.40],
    'fold_3': [0.30, 0.40, 0.50],
}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()
WALK_FORWARD_SPECS = [
    {
        'fold': 'fold_1',
        'train_start': 0.00,
        'train_end': 0.30,
        'calibration_start': 0.30,
        'calibration_end': 0.40,
        'evaluation_start': 0.40,
        'evaluation_end': 0.50,
    },
    {
        'fold': 'fold_2',
        'train_start': 0.00,
        'train_end': 0.40,
        'calibration_start': 0.40,
        'calibration_end': 0.50,
        'evaluation_start': 0.50,
        'evaluation_end': 0.60,
    },
    {
        'fold': 'fold_3',
        'train_start': 0.00,
        'train_end': 0.50,
        'calibration_start': 0.50,
        'calibration_end': 0.60,
        'evaluation_start': 0.60,
        'evaluation_end': 0.70,
    },
]
XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'n_estimators': 400,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 5.0,
    'max_delta_step': 1.0,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': 0,
}
MODEL_SPECS = [
    {
        'model_id': '003_004_single_unweighted',
        'label': '003/004 single type-expert',
        'kind': 'single',
        'use_class_weight': False,
    },
    {
        'model_id': '005_expanding_fold_ensemble',
        'label': '005 expanding checkpoint ensemble',
        'kind': 'ensemble',
        'use_class_weight': False,
    },
    {
        'model_id': '007_single_class_weight',
        'label': '007 single type-expert + scale_pos_weight',
        'kind': 'single',
        'use_class_weight': True,
    },
]

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


def compute_scale_pos_weight(y: pd.Series) -> float:
    positive = int(y.sum())
    negative = int(len(y) - positive)
    if positive == 0 or negative == 0:
        raise ValueError('scale_pos_weight는 양성과 음성이 모두 있는 Train에서만 계산할 수 있습니다.')
    return negative / positive


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'notebooks').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.')


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / 'data' / 'raw',
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / 'dataset.csv'
        mapping_path = resolved / 'mapping.json'
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError('dataset.csv와 mapping.json 쌍을 찾지 못했습니다.')


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
REPORT_PATH = REPO_ROOT / 'docs' / 'experiments' / f'{EXPERIMENT_ID}.md'
LOG_DIR = REPO_ROOT / 'docs' / 'peace'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f'{EXPERIMENT_ID}.log'

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler = logging.FileHandler(LOG_PATH, mode='w', encoding='utf-8')
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info('experiment=%s', EXPERIMENT_ID)
logger.info('data_file=%s sha256=%s', DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info('mapping_file=%s sha256=%s', MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    'versions python=%s pandas=%s sklearn=%s xgboost=%s',
    sys.version.split()[0],
    pd.__version__,
    sklearn.__version__,
    xgboost.__version__,
)
logger.info('target_recalls=%s diagnostic_target=%.2f', TARGET_RECALLS, DIAGNOSTIC_TARGET)
print('log saved to:', LOG_PATH.relative_to(REPO_ROOT))

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith('Unnamed:') or source_index_column == '':
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f'예상하지 못한 첫 번째 컬럼: {source_index_column}')

with MAPPING_PATH.open(encoding='utf-8') as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f'필수 컬럼 누락: {sorted(missing_required)}'
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {'0', '1', '2', '3', '4'}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors='raise', utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind='stable').reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith('inspection_feat')]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith('meta_feat')]
feature_columns_by_type = {}
feature_rows = []
for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            'inspection_type': inspection_type,
            'meta_features': len(meta_columns),
            'mapped_inspection_features': len(mapped_columns),
            'raw_features_used': len(selected_columns),
        }
    )
feature_summary = pd.DataFrame(feature_rows).set_index('inspection_type')

data_summary = pd.Series(
    {
        'rows': len(raw_df),
        'columns': raw_df.shape[1],
        'false_call_0': int((raw_df[TARGET] == 0).sum()),
        'real_defect_1': int((raw_df[TARGET] == 1).sum()),
        'real_defect_rate_pct': raw_df[TARGET].mean() * 100,
        'inspection_types': raw_df[TYPE_COLUMN].nunique(),
        'inspection_features': len(inspection_columns),
        'mapped_feature_union': len(mapped_union),
        'timestamp_start': raw_df[TIME_COLUMN].min(),
        'timestamp_end': raw_df[TIME_COLUMN].max(),
    },
    name='raw_data',
)
display(data_summary)
display(feature_summary)
logger.info('data_verified rows=%d columns=%d class_0=%d class_1=%d', len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum()))

timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side='left'))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (raw_df[TIME_COLUMN] > train_end_time) & (raw_df[TIME_COLUMN] <= validation_end_time)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            'split': 'train',
            'rows': len(train_df),
            'positive_samples': int(train_df[TARGET].sum()),
            'positive_rate_pct': train_df[TARGET].mean() * 100,
            'timestamp_groups': train_df[TIME_COLUMN].nunique(),
            'start_time': train_df[TIME_COLUMN].min(),
            'end_time': train_df[TIME_COLUMN].max(),
        },
        {
            'split': 'validation',
            'rows': len(validation_df),
            'positive_samples': int(validation_df[TARGET].sum()),
            'positive_rate_pct': validation_df[TARGET].mean() * 100,
            'timestamp_groups': validation_df[TIME_COLUMN].nunique(),
            'start_time': validation_df[TIME_COLUMN].min(),
            'end_time': validation_df[TIME_COLUMN].max(),
        },
        {
            'split': 'test',
            'rows': len(test_df),
            'positive_samples': int(test_df[TARGET].sum()),
            'positive_rate_pct': test_df[TARGET].mean() * 100,
            'timestamp_groups': test_df[TIME_COLUMN].nunique(),
            'start_time': test_df[TIME_COLUMN].min(),
            'end_time': test_df[TIME_COLUMN].max(),
        },
    ]
).set_index('split')
display(split_summary)
logger.info('split_summary=%s', split_summary.reset_index().to_dict(orient='records'))

walk_forward_boundaries = {fraction: boundary_at(fraction) for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]}
walk_forward_segments = {}
walk_forward_split_rows = []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec['fold']
    train_end = walk_forward_boundaries[spec['train_end']]
    calibration_start = walk_forward_boundaries[spec['calibration_start']]
    calibration_end = walk_forward_boundaries[spec['calibration_end']]
    evaluation_start = walk_forward_boundaries[spec['evaluation_start']]
    evaluation_end = walk_forward_boundaries[spec['evaluation_end']]
    segments = {
        'train': raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        'calibration': raw_df.loc[(raw_df[TIME_COLUMN] > calibration_start) & (raw_df[TIME_COLUMN] <= calibration_end)],
        'evaluation': raw_df.loc[(raw_df[TIME_COLUMN] > evaluation_start) & (raw_df[TIME_COLUMN] <= evaluation_end)],
    }
    assert segments['train'][TIME_COLUMN].max() < segments['calibration'][TIME_COLUMN].min()
    assert segments['calibration'][TIME_COLUMN].max() < segments['evaluation'][TIME_COLUMN].min()
    walk_forward_segments[fold_name] = segments
    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                'fold': fold_name,
                'segment': segment_name,
                'rows': len(frame),
                'positive_samples': int(frame[TARGET].sum()),
                'positive_rate_pct': frame[TARGET].mean() * 100,
                'timestamp_groups': frame[TIME_COLUMN].nunique(),
                'start_time': frame[TIME_COLUMN].min(),
                'end_time': frame[TIME_COLUMN].max(),
            }
        )
walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(['fold', 'segment'])
display(walk_forward_split_summary)
logger.info('walk_forward_split_summary=%s', walk_forward_split_summary.reset_index().to_dict(orient='records'))


def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        'rows': len(y_true),
        'positive_samples': int(y_true.sum()),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        'false_call_reduction': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        'roc_auc': roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        'pr_auc': average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall):
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError('임계값 선택에는 positive와 negative가 모두 필요합니다.')
    order = np.argsort(-probability, kind='stable')
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(np.r_[sorted_probability[:-1] != sorted_probability[1:], True])
    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f'Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.')
    best_local = np.lexsort((thresholds[feasible], recall[feasible], false_call_reduction[feasible]))[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {'threshold': selected_threshold, 'min_recall': min_recall, **metrics}


_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics['recall'] >= 2 / 3:
        _reference_rows.append((_metrics['false_call_reduction'], _metrics['recall'], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized['threshold'], _reference[2])
logger.info('threshold_selector_unit_test=PASS')


def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            ('categorical', OneHotEncoder(handle_unknown='ignore', dtype=np.float32), categorical),
            ('continuous', 'passthrough', continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def predict_with_type_experts(train_frame, target_frames, use_class_weight, model_id, stage_name):
    prediction_map = {
        target_name: pd.Series(np.nan, index=frame.index, dtype='float64')
        for target_name, frame in target_frames.items()
    }
    training_rows = []
    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype('int8')
        assert len(type_train) > 0
        assert y_train.nunique() == 2
        scale_pos_weight = compute_scale_pos_weight(y_train) if use_class_weight else 1.0
        logger.info(
            'fit_start model=%s stage=%s type=%d train_rows=%d train_positive=%d scale_pos_weight=%.6f',
            model_id,
            stage_name,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            scale_pos_weight,
        )
        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        model_params = XGB_PARAMS.copy()
        if use_class_weight:
            model_params['scale_pos_weight'] = scale_pos_weight
        model = XGBClassifier(**model_params)
        model.fit(X_train, y_train, verbose=False)
        for target_name, frame in target_frames.items():
            type_target = frame.loc[frame[TYPE_COLUMN] == inspection_type]
            assert len(type_target) > 0
            X_target = preprocessor.transform(type_target[feature_columns])
            probability = model.predict_proba(X_target)[:, 1]
            prediction_map[target_name].loc[type_target.index] = probability
            del X_target, probability
        training_rows.append(
            {
                'model_id': model_id,
                'stage': stage_name,
                'inspection_type': inspection_type,
                'train_rows': len(type_train),
                'train_positive': int(y_train.sum()),
                'raw_features': len(feature_columns),
                'encoded_features': X_train.shape[1],
                'scale_pos_weight': scale_pos_weight,
            }
        )
        logger.info('fit_done model=%s stage=%s type=%d', model_id, stage_name, inspection_type)
        del preprocessor, model, X_train
        gc.collect()
    for target_name, probability in prediction_map.items():
        assert probability.notna().all(), (model_id, stage_name, target_name)
    return prediction_map, training_rows


def evaluate_recall_target(model_id, model_label, fold_name, calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, target_recall):
    selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=target_recall)
    future_metrics = evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, selection['threshold'])
    row = {
        'model_id': model_id,
        'model_label': model_label,
        'fold': fold_name,
        'target_recall': target_recall,
        'calibration_rows': int(selection['rows']),
        'calibration_positive_samples': int(selection['positive_samples']),
        'calibration_threshold': float(selection['threshold']),
        'calibration_recall': float(selection['recall']),
        'calibration_fp': int(selection['fp']),
        'calibration_fn': int(selection['fn']),
        'calibration_fcr': float(selection['false_call_reduction']),
        'future_rows': int(future_metrics['rows']),
        'future_positive_samples': int(future_metrics['positive_samples']),
        'future_recall': float(future_metrics['recall']),
        'future_fp': int(future_metrics['fp']),
        'future_fn': int(future_metrics['fn']),
        'future_fcr': float(future_metrics['false_call_reduction']),
        'future_precision': float(future_metrics['precision']),
        'future_pr_auc': float(future_metrics['pr_auc']),
        'future_tp': int(future_metrics['tp']),
        'future_tn': int(future_metrics['tn']),
    }
    prediction = (np.asarray(evaluation_probability) >= selection['threshold']).astype(np.int8)
    prediction_frame = pd.DataFrame(
        {
            RECORD_ID: evaluation_frame[RECORD_ID].to_numpy(),
            'fold': fold_name,
            'model_id': model_id,
            'target_recall': target_recall,
            'y_true': evaluation_frame[TARGET].astype('int8').to_numpy(),
            'probability': np.asarray(evaluation_probability, dtype=np.float64),
            'prediction': prediction,
        }
    )
    return row, prediction_frame


walk_probability_store = {}
training_history_rows = []
for spec in MODEL_SPECS:
    model_id = spec['model_id']
    walk_probability_store[model_id] = {}
    if spec['kind'] == 'single':
        for fold_spec in WALK_FORWARD_SPECS:
            fold_name = fold_spec['fold']
            segments = walk_forward_segments[fold_name]
            target_frames = {
                'calibration': segments['calibration'],
                'evaluation': segments['evaluation'],
            }
            prediction_map, training_rows = predict_with_type_experts(
                segments['train'],
                target_frames,
                use_class_weight=spec['use_class_weight'],
                model_id=model_id,
                stage_name=fold_name,
            )
            walk_probability_store[model_id][fold_name] = prediction_map
            training_history_rows.extend(training_rows)
    else:
        prediction_targets = {}
        for fold_spec in WALK_FORWARD_SPECS:
            fold_name = fold_spec['fold']
            prediction_targets[f'{fold_name}_calibration'] = walk_forward_segments[fold_name]['calibration']
            prediction_targets[f'{fold_name}_evaluation'] = walk_forward_segments[fold_name]['evaluation']
        checkpoint_predictions = {
            checkpoint: {
                target_name: pd.Series(np.nan, index=frame.index, dtype='float64')
                for target_name, frame in prediction_targets.items()
                if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
            }
            for checkpoint in ENSEMBLE_CHECKPOINTS
        }
        for checkpoint in ENSEMBLE_CHECKPOINTS:
            checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
            target_frames = {name: prediction_targets[name] for name in checkpoint_predictions[checkpoint]}
            prediction_map, training_rows = predict_with_type_experts(
                checkpoint_train,
                target_frames,
                use_class_weight=False,
                model_id=model_id,
                stage_name=f'checkpoint_{checkpoint:.2f}',
            )
            checkpoint_predictions[checkpoint] = prediction_map
            training_history_rows.extend(training_rows)
        for fold_name, members in FOLD_MEMBER_CHECKPOINTS.items():
            calibration_name = f'{fold_name}_calibration'
            evaluation_name = f'{fold_name}_evaluation'
            calibration_probability = pd.Series(
                np.mean(np.vstack([checkpoint_predictions[checkpoint][calibration_name].to_numpy() for checkpoint in members]), axis=0),
                index=walk_forward_segments[fold_name]['calibration'].index,
                dtype='float64',
            )
            evaluation_probability = pd.Series(
                np.mean(np.vstack([checkpoint_predictions[checkpoint][evaluation_name].to_numpy() for checkpoint in members]), axis=0),
                index=walk_forward_segments[fold_name]['evaluation'].index,
                dtype='float64',
            )
            walk_probability_store[model_id][fold_name] = {
                'calibration': calibration_probability,
                'evaluation': evaluation_probability,
            }

walk_rows = []
walk_prediction_frames = []
for spec in MODEL_SPECS:
    model_id = spec['model_id']
    model_label = spec['label']
    for fold_spec in WALK_FORWARD_SPECS:
        fold_name = fold_spec['fold']
        segments = walk_forward_segments[fold_name]
        calibration_probability = walk_probability_store[model_id][fold_name]['calibration']
        evaluation_probability = walk_probability_store[model_id][fold_name]['evaluation']
        for target_recall in TARGET_RECALLS:
            row, prediction_frame = evaluate_recall_target(
                model_id,
                model_label,
                fold_name,
                segments['calibration'],
                calibration_probability,
                segments['evaluation'],
                evaluation_probability,
                target_recall,
            )
            walk_rows.append(row)
            if np.isclose(target_recall, DIAGNOSTIC_TARGET):
                walk_prediction_frames.append(prediction_frame)
        logger.info('walk_evaluated model=%s fold=%s', model_id, fold_name)

walk_results = pd.DataFrame(walk_rows).sort_values(['target_recall', 'model_id', 'fold']).reset_index(drop=True)
training_history = pd.DataFrame(training_history_rows)

def summarize_walk(group: pd.DataFrame) -> pd.Series:
    target_recall = float(group['target_recall'].iloc[0])
    return pd.Series(
        {
            'folds': int(group['fold'].nunique()),
            'mean_calibration_threshold': group['calibration_threshold'].mean(),
            'mean_future_recall': group['future_recall'].mean(),
            'min_future_recall': group['future_recall'].min(),
            'recall_target_hit_folds': int((group['future_recall'] >= target_recall).sum()),
            'mean_future_fp': group['future_fp'].mean(),
            'total_future_fp': int(group['future_fp'].sum()),
            'mean_future_fn': group['future_fn'].mean(),
            'mean_future_fcr': group['future_fcr'].mean(),
        }
    )

walk_summary = walk_results.groupby(['target_recall', 'model_id', 'model_label'], sort=True).apply(summarize_walk).reset_index()
walk_summary = walk_summary.sort_values(['target_recall', 'mean_future_fp', 'mean_future_fcr'], ascending=[True, True, False]).reset_index(drop=True)
display(walk_results)
display(walk_summary)
logger.info('walk_summary=%s', walk_summary.to_dict(orient='records'))

final_probability_store = {}
for spec in MODEL_SPECS:
    model_id = spec['model_id']
    if spec['kind'] == 'single':
        prediction_map, training_rows = predict_with_type_experts(
            train_df,
            {'validation': validation_df, 'test': test_df},
            use_class_weight=spec['use_class_weight'],
            model_id=model_id,
            stage_name='final_train70',
        )
        final_probability_store[model_id] = prediction_map
        training_history = pd.concat([training_history, pd.DataFrame(training_rows)], ignore_index=True)
    else:
        prediction_targets = {
            'final_validation': validation_df,
            'final_test': test_df,
        }
        checkpoint_predictions = {}
        for checkpoint in FINAL_MEMBER_CHECKPOINTS:
            checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
            prediction_map, training_rows = predict_with_type_experts(
                checkpoint_train,
                prediction_targets,
                use_class_weight=False,
                model_id=model_id,
                stage_name=f'final_checkpoint_{checkpoint:.2f}',
            )
            checkpoint_predictions[checkpoint] = prediction_map
            training_history = pd.concat([training_history, pd.DataFrame(training_rows)], ignore_index=True)
        validation_probability = pd.Series(
            np.mean(np.vstack([checkpoint_predictions[checkpoint]['final_validation'].to_numpy() for checkpoint in FINAL_MEMBER_CHECKPOINTS]), axis=0),
            index=validation_df.index,
            dtype='float64',
        )
        test_probability = pd.Series(
            np.mean(np.vstack([checkpoint_predictions[checkpoint]['final_test'].to_numpy() for checkpoint in FINAL_MEMBER_CHECKPOINTS]), axis=0),
            index=test_df.index,
            dtype='float64',
        )
        final_probability_store[model_id] = {
            'validation': validation_probability,
            'test': test_probability,
        }

final_rows = []
for spec in MODEL_SPECS:
    model_id = spec['model_id']
    model_label = spec['label']
    validation_probability = final_probability_store[model_id]['validation']
    test_probability = final_probability_store[model_id]['test']
    for target_recall in TARGET_RECALLS:
        selection = select_threshold(validation_df[TARGET], validation_probability, min_recall=target_recall)
        test_metrics = evaluate_probabilities(test_df[TARGET], test_probability, selection['threshold'])
        final_rows.append(
            {
                'model_id': model_id,
                'model_label': model_label,
                'target_recall': target_recall,
                'validation_threshold': float(selection['threshold']),
                'validation_recall': float(selection['recall']),
                'validation_fp': int(selection['fp']),
                'validation_fcr': float(selection['false_call_reduction']),
                'test_recall': float(test_metrics['recall']),
                'test_fp': int(test_metrics['fp']),
                'test_fn': int(test_metrics['fn']),
                'test_fcr': float(test_metrics['false_call_reduction']),
                'test_precision': float(test_metrics['precision']),
                'test_pr_auc': float(test_metrics['pr_auc']),
                'test_tp': int(test_metrics['tp']),
                'test_tn': int(test_metrics['tn']),
                'selection_note': 'Test reused from prior experiments; threshold selected on validation only.',
            }
        )
final_results = pd.DataFrame(final_rows).sort_values(['target_recall', 'test_fp', 'test_fcr'], ascending=[True, True, False]).reset_index(drop=True)
display(final_results)
logger.info('final_results=%s', final_results.to_dict(orient='records'))

pooled_predictions = pd.concat(walk_prediction_frames, ignore_index=True)
diagnostic_rows = []
correlation_rows = []
for model_a, model_b in combinations([spec['model_id'] for spec in MODEL_SPECS], 2):
    pair = pooled_predictions.pivot_table(
        index=[RECORD_ID, 'fold', 'target_recall', 'y_true'],
        columns='model_id',
        values=['prediction', 'probability'],
    )
    pred_a = pair[('prediction', model_a)].astype('int8')
    pred_b = pair[('prediction', model_b)].astype('int8')
    prob_a = pair[('probability', model_a)].astype('float64')
    prob_b = pair[('probability', model_b)].astype('float64')
    y_true = pair.index.get_level_values('y_true').astype('int8')
    fn_a = (y_true == 1) & (pred_a == 0)
    fn_b = (y_true == 1) & (pred_b == 0)
    fp_a = (y_true == 0) & (pred_a == 1)
    fp_b = (y_true == 0) & (pred_b == 1)
    diagnostic_rows.append(
        {
            'target_recall': DIAGNOSTIC_TARGET,
            'model_a': model_a,
            'model_b': model_b,
            'disagreement_rate': float((pred_a != pred_b).mean()),
            'probability_corr': float(np.corrcoef(prob_a, prob_b)[0, 1]),
            'fn_a': int(fn_a.sum()),
            'fn_b': int(fn_b.sum()),
            'shared_fn': int((fn_a & fn_b).sum()),
            'fn_only_a': int((fn_a & ~fn_b).sum()),
            'fn_only_b': int((fn_b & ~fn_a).sum()),
            'fp_a': int(fp_a.sum()),
            'fp_b': int(fp_b.sum()),
            'shared_fp': int((fp_a & fp_b).sum()),
            'fp_only_a': int((fp_a & ~fp_b).sum()),
            'fp_only_b': int((fp_b & ~fp_a).sum()),
        }
    )

diagnostic_summary = pd.DataFrame(diagnostic_rows).sort_values(['model_a', 'model_b']).reset_index(drop=True)
display(diagnostic_summary)
logger.info('diagnostic_summary=%s', diagnostic_summary.to_dict(orient='records'))

DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        'dataset_sha256_unchanged': True,
        'mapping_sha256_unchanged': True,
        'dataset_sha256': DATA_SHA256_BEFORE,
        'mapping_sha256': MAPPING_SHA256_BEFORE,
        'walk_rows': len(walk_results),
        'final_rows': len(final_results),
        'diagnostic_pairs': len(diagnostic_summary),
        'log_file': f'docs/peace/{LOG_PATH.name}',
        'report_file': f'docs/experiments/{REPORT_PATH.name}',
    },
    name='verification',
)
display(verification)
logger.info('source_integrity=PASS')

MODEL_LABELS = {spec['model_id']: spec['label'] for spec in MODEL_SPECS}


def pct(value: float) -> str:
    return f'{value * 100:.2f}%'


def md_table(df: pd.DataFrame, float_formats=None) -> str:
    if float_formats is None:
        float_formats = {}
    formatted = df.copy()
    for column, formatter in float_formats.items():
        if column in formatted.columns:
            formatted[column] = formatted[column].map(formatter)
    return formatted.to_markdown(index=False)


walk_report = walk_results[
    [
        'target_recall',
        'model_label',
        'fold',
        'calibration_threshold',
        'calibration_fp',
        'calibration_fcr',
        'future_recall',
        'future_fp',
        'future_fcr',
        'future_fn',
    ]
].rename(
    columns={
        'target_recall': 'target_recall',
        'model_label': 'model',
        'fold': 'fold',
        'calibration_threshold': 'calibration_threshold',
        'calibration_fp': 'calibration_fp',
        'calibration_fcr': 'calibration_fcr',
        'future_recall': 'future_recall',
        'future_fp': 'future_fp',
        'future_fcr': 'future_fcr',
        'future_fn': 'future_fn',
    }
)
walk_summary_report = walk_summary[
    [
        'target_recall',
        'model_label',
        'mean_future_recall',
        'min_future_recall',
        'recall_target_hit_folds',
        'mean_future_fp',
        'total_future_fp',
        'mean_future_fcr',
    ]
].rename(columns={'model_label': 'model'})
final_report = final_results[
    [
        'target_recall',
        'model_label',
        'validation_threshold',
        'validation_fp',
        'validation_fcr',
        'test_recall',
        'test_fp',
        'test_fcr',
        'test_fn',
        'selection_note',
    ]
].rename(columns={'model_label': 'model'})
diagnostic_report = diagnostic_summary.copy()
diagnostic_report['model_a'] = diagnostic_report['model_a'].map(MODEL_LABELS)
diagnostic_report['model_b'] = diagnostic_report['model_b'].map(MODEL_LABELS)

best_by_target = final_results.sort_values(['target_recall', 'test_fp', 'test_fcr'], ascending=[True, True, False]).groupby('target_recall', sort=True).first().reset_index()
conclusion_lines = []
for _, row in best_by_target.iterrows():
    conclusion_lines.append(
        f"- Target recall {row['target_recall']:.2f}: lowest Test FP came from {row['model_label']} at threshold {row['validation_threshold']:.6f} with Test recall {pct(row['test_recall'])}, FP {int(row['test_fp']):,}, FCR {pct(row['test_fcr'])}."
    )
ensemble_diag = diagnostic_summary[
    (diagnostic_summary['model_a'] == '005_expanding_fold_ensemble')
    | (diagnostic_summary['model_b'] == '005_expanding_fold_ensemble')
].copy()
if not ensemble_diag.empty:
    diag_line = ensemble_diag.sort_values('disagreement_rate', ascending=False).iloc[0]
    other_model = diag_line['model_b'] if diag_line['model_a'] == '005_expanding_fold_ensemble' else diag_line['model_a']
    conclusion_lines.append(
        f"- Ensemble diagnostic at target recall {DIAGNOSTIC_TARGET:.2f}: versus {MODEL_LABELS[other_model]}, disagreement was {pct(diag_line['disagreement_rate'])}, shared FN {int(diag_line['shared_fn'])}, ensemble-only FN {int(diag_line['fn_only_a'] if diag_line['model_a'] == '005_expanding_fold_ensemble' else diag_line['fn_only_b'])}, ensemble-only FP {int(diag_line['fp_only_a'] if diag_line['model_a'] == '005_expanding_fold_ensemble' else diag_line['fp_only_b'])}."
    )

report_text = "\n".join([
    f'# {EXPERIMENT_ID}',
    '',
    '## 연결된 노트북',
    '',
    f'`notebooks/{EXPERIMENT_ID}.ipynb`',
    '',
    '## 상태',
    '',
    '완료',
    '',
    '## 목적',
    '',
    '003/004 단일 type-expert, 005 expanding checkpoint ensemble, 007 type별 `scale_pos_weight` 모델을 동일한 분할·피처·XGBoost 파라미터에서 비교하고, 동일 Recall 목표에서 미래 Fold와 최종 Test의 FP/FCR 차이를 정리한다.',
    '',
    '## 설정',
    '',
    f'- Target recall: {TARGET_RECALLS}',
    f'- Walk-forward fold: 004/005와 동일한 expanding 3-fold',
    f'- Final benchmark: 0~70% Train / 70~80% Validation / 80~100% Test',
    f'- Threshold rule: 기존 `select_threshold` 로직 그대로 사용, calibration/validation에서 recall 제약을 만족하는 threshold 중 FCR 최대 선택',
    f'- Source integrity: dataset sha256 `{DATA_SHA256_BEFORE}`, mapping sha256 `{MAPPING_SHA256_BEFORE}`',
    f'- Log: `docs/peace/{LOG_PATH.name}`',
    '',
    '## 데이터 분할',
    '',
    split_summary.reset_index().to_markdown(index=False),
    '',
    '## Walk-forward 구성',
    '',
    walk_forward_split_summary.reset_index().to_markdown(index=False),
    '',
    '## Fold별 Recall-aligned 결과',
    '',
    md_table(
        walk_report,
        {
            'target_recall': lambda v: f'{v:.2f}',
            'calibration_threshold': lambda v: f'{v:.6f}',
            'calibration_fcr': pct,
            'future_recall': pct,
            'future_fcr': pct,
        },
    ),
    '',
    '## Walk-forward 요약',
    '',
    md_table(
        walk_summary_report,
        {
            'target_recall': lambda v: f'{v:.2f}',
            'mean_future_recall': pct,
            'min_future_recall': pct,
            'mean_future_fp': lambda v: f'{v:,.1f}',
            'mean_future_fcr': pct,
        },
    ),
    '',
    '## Final Retrospective Benchmark',
    '',
    '아래 Test는 이전 실험들과 같은 80~100% 구간이며, 이번 실험에서도 threshold 선택에는 사용하지 않았다.',
    '',
    md_table(
        final_report,
        {
            'target_recall': lambda v: f'{v:.2f}',
            'validation_threshold': lambda v: f'{v:.6f}',
            'validation_fcr': pct,
            'test_recall': pct,
            'test_fcr': pct,
        },
    ),
    '',
    f'## Prediction Disagreement / FN Overlap Diagnostic (target recall {DIAGNOSTIC_TARGET:.2f})',
    '',
    md_table(
        diagnostic_report,
        {
            'target_recall': lambda v: f'{v:.2f}',
            'disagreement_rate': pct,
            'probability_corr': lambda v: f'{v:.4f}',
        },
    ),
    '',
    '## 결론',
    '',
    *conclusion_lines,
    '',
    '## 실행 로그',
    '',
    f'`docs/peace/{LOG_PATH.name}`',
])
REPORT_PATH.write_text(report_text + '\n', encoding='utf-8')
display(Markdown(report_text[:4000]))
print(f'report saved to: {REPORT_PATH.relative_to(REPO_ROOT)}')
logger.info('report_saved=%s', REPORT_PATH.name)
logger.info('experiment_complete=%s', EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


2026-08-25 14:27:38,237 | INFO | experiment=0825_peace_012_recall_aligned_model_comparison


2026-08-25 14:27:38,237 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 14:27:38,238 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 14:27:38,238 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 14:27:38,239 | INFO | target_recalls=[0.95, 0.97, 0.99] diagnostic_target=0.97


...

dataset_sha256_unchanged                                                 True
mapping_sha256_unchanged                                                 True
dataset_sha256              53e8568743216d556856ed69b388f6750fbfa0b8c59ad3...
mapping_sha256              3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a...
walk_rows                                                                  27
final_rows                                                                  9
diagnostic_pairs                                                            3
log_file                    docs/peace/0825_peace_012_recall_aligned_model...
report_file                 docs/experiments/0825_peace_012_recall_aligned...
Name: verification, dtype: object

2026-08-25 14:28:35,215 | INFO | source_integrity=PASS


# 0825_peace_012_recall_aligned_model_comparison

## 연결된 노트북

`notebooks/0825_peace_012_recall_aligned_model_comparison.ipynb`

## 상태

완료

## 목적

003/004 단일 type-expert, 005 expanding checkpoint ensemble, 007 type별 `scale_pos_weight` 모델을 동일한 분할·피처·XGBoost 파라미터에서 비교하고, 동일 Recall 목표에서 미래 Fold와 최종 Test의 FP/FCR 차이를 정리한다.

## 설정

- Target recall: [0.95, 0.97, 0.99]
- Walk-forward fold: 004/005와 동일한 expanding 3-fold
- Final benchmark: 0~70% Train / 70~80% Validation / 80~100% Test
- Threshold rule: 기존 `select_threshold` 로직 그대로 사용, calibration/validation에서 recall 제약을 만족하는 threshold 중 FCR 최대 선택
- Source integrity: dataset sha256 `53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62`, mapping sha256 `3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486`
- Log: `docs/peace/0825_peace_012_recall_aligned_model_comparison.log`

## 데이터 분할

| split      |   rows |   positive_samples |   positive_rate_pct |   timestamp_groups | start_time                | end_time                  |
|:-----------|-------:|-------------------:|--------------------:|-------------------:|:--------------------------|:--------------------------|
| train      | 308196 |               1940 |            0.62947  |              29249 | 1970-06-23 03:58:55+00:00 | 1970-10-05 00:29:59+00:00 |
| validation |  44026 |                357 |            0.810884 |               3400 | 1970-10-05 00:30:30+00:00 | 1970-10-13 16:54:14+00:00 |
| test       |  88052 |               2325 |            2.64049  |               7093 | 1970-10-13 16:54:52+00:00 | 1970-11-02 14:21:28+00:00 |

## Walk-forward 구성

| fold   | segment     |   rows |   positive_samples |   positive_rate_pct |   timestamp_groups | start_time                | end_time                  |
|:-------|:------------|-------:|-------------------:|--------------------:|-------------------:|:--------------------------|:--------------------------|
| fold_1 | train       | 132137 |               1223 |           0.925555  |              15230 | 1970-06-23 03:58:55+00:00 | 1970-08-18 06:51:10+00:00 |
| fold_1 | calibration |  43979 |                200 |           0.454763  |               1251 | 1970-08-18 06:51:41+00:00 | 1970-08-21 23:32:59+00:00 |
| fold_1 | evaluation  |  44040 |                326 |           0.740236  |               5415 | 1970-08-21 23:33:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_2 | train       | 176116 |               1423 |           0.80799   |              16481 | 1970-06-23 03:58:55+00:00 | 1970-08-21 23:32:59+00:00 |
| fold_2 | calibration |  44040 |                326 |           0.740236  |               5415 | 1970-08-21 23:33:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_2 | evaluation  |  44187 |                152 |           0.343993  |               4167 | 1970-09-15 06:47:13+00:00 | 1970-09-28 05:10:37+00:00 |
| fold_3 | train       | 220156 |               1749 |           0.794437  |              21896 | 1970-06-23 03:58:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_3 | calibration |  44187 |                152 |           0.343993  |               4167 | 1970-09-15 06:47:13+00:00 | 1970-09-28 05:10:37+00:00 |
| fold_3 | evaluation  |  43853 |                 39 |           0.0889335 |               3186 | 1970-09-28 05:11:13+00:00 | 1970-10-05 00:29:59+00:00 |

## Fold별 Recall-aligned 결과

|   target_recall | model                                     | fold   |   calibration_threshold |   calibration_fp | calibration_fcr   | future_recall   |   future_fp | future_fcr   |   future_fn |
|----------------:|:------------------------------------------|:-------|------------------------:|-----------------:|:------------------|:----------------|------------:|:-------------|------------:|
|            0.95 | 003/004 single type-expert                | fold_1 |                0.000415 |            31728 | 27.53%            | 98.77%          |       34809 | 20.37%       |           4 |
|            0.95 | 003/004 single type-expert                | fold_2 |     

report saved to: docs/experiments/0825_peace_012_recall_aligned_model_comparison.md
2026-08-25 14:28:35,229 | INFO | report_saved=0825_peace_012_recall_aligned_model_comparison.md


2026-08-25 14:28:35,229 | INFO | experiment_complete=0825_peace_012_recall_aligned_model_comparison
